[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/02_opacity_photosphere.ipynb)

# 2 · Opacity and the photosphere

*Brown Dwarf Spectra On-Ramp · about 45 minutes*

**By the end of this notebook you will be able to:**
- describe a brown dwarf atmosphere as a temperature–pressure profile with no surface
- explain optical depth and the τ ≈ 2/3 photosphere, and what the flux at a wavelength tells you about temperature
- explain why optical depth depends on wavelength, and what a band's core and wings are
- read a contribution function
- predict the spectrum of an isothermal atmosphere and of one with a thermal inversion
- explain why an absorption band stops deepening (saturates)

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.
> **Moving between notebooks:** use the links inside the notebooks (such as **Next** at the bottom). They always open
> the original notebook, so they work from your copy too — then save a copy of the new notebook before you edit it.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

## 1. No surface, just layers

A brown dwarf is a ball of gas — mostly hydrogen and helium — with **no solid surface**. Going down,
the pressure and temperature rise smoothly. We describe the atmosphere by its **temperature–pressure
(T–P) profile**. Pressure is the natural "height" coordinate: low pressure = high up.

> 🧸 **TOY MODEL.** The atmosphere in this notebook is a deliberately simple toy. Its bands sit at roughly the right wavelengths so the plots look familiar, but their shapes and strengths are made up. Use it for *ideas*, never for numbers about real objects.

In [ ]:
std, iso, inv = bd.load_toy("standard"), bd.load_toy("isothermal"), bd.load_toy("inversion")
print(std.meta["label"]); print(std.meta["description"])
fig, ax = plt.subplots(figsize=(5, 6))
for t, c in [(std, bd.LINE_COLORS[0]), (iso, bd.LINE_COLORS[1]), (inv, bd.LINE_COLORS[2])]:
    ax.plot(t.temperature_k, t.pressure_bar, color=c, label=t.name)
ax.set_yscale("log"); ax.invert_yaxis()
ax.set_xlabel("Temperature [K]"); ax.set_ylabel("Pressure [bar]  (up = higher)")
ax.legend(); ax.set_title("TOY temperature-pressure profiles", fontsize=10);

## 2. Optical depth and the photosphere

Light trying to escape from deep down gets absorbed on the way out. The **optical depth** τ counts how
many "absorption lengths" lie between a layer and space: light from a layer at depth τ escapes with
probability about e<sup>−τ</sup>. So:

* layers with τ ≫ 1 are hidden;
* layers with τ ≪ 1 emit too little to matter (too little gas);
* most escaping light comes from around **τ ≈ 2/3** — call that the **photosphere**.

**Why τ depends on wavelength.** Each kind of atom or molecule absorbs light only at certain wavelengths, set by the
ways it can rotate and vibrate. Water, for example, absorbs strongly near 1.4 µm but only weakly near 1.27 µm. A
range of wavelengths where a molecule absorbs strongly is called a **band**. So τ depends on wavelength: inside a
band, τ = 2/3 is reached *high up*; in a **window** between bands, where little absorbs, we see *deep*.

**What the flux tells you:** the flux at each wavelength tells you the approximate **temperature of the layer
where τ = 2/3** at that wavelength. A hot layer gives a lot of flux; a cool layer gives little.

**Pressure as depth.** Atmospheric scientists use **pressure as a stand-in for depth**: pressure rises steadily as
you go down, so **higher pressure = deeper in the atmosphere**. The plots below put high pressure at the bottom, so
down on the plot is down into the atmosphere.

In [ ]:
fig, ax = plt.subplots()
ax.plot(std.wave_um, std.photosphere_bar, color="k")
ax.set_yscale("log"); ax.invert_yaxis()
ax.set_xlabel("Wavelength [µm]"); ax.set_ylabel("pressure where τ = 2/3 [bar]\n(higher pressure = deeper)")
ax.set_title("TOY: where each wavelength's light comes from", fontsize=10);

### A band's core and wings
A band absorbs most strongly at its **core**. Away from the core the absorption gets weaker, but it does not stop
abruptly: the weaker absorption on either side is the band's **wings**, and it can stretch well into a neighbouring
window. Wings come mainly from **pressure broadening**: molecules that collide often — at high pressure, deep in the
atmosphere — absorb over a wider range of wavelengths, so the band spreads out.

Here is a single made-up band at two pressures. The top panel shows how strongly it absorbs (its optical depth); the
bottom panel shows what fraction of the light gets through.

> 🧸 **TOY** — one invented band, to show the shapes. Real bands are many overlapping lines.

In [ ]:
lam = np.linspace(1.0, 2.0, 2000)          # microns
centre = 1.40

def toy_band(width, wing=0.01):
    x = (lam - centre) / width
    return np.exp(-0.5 * x**2) + wing / (1 + x**2)     # strong core + weak, wide wings

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for width, label, c in [(0.03, "low pressure", bd.LINE_COLORS[1]), (0.06, "high pressure (broadened)", bd.LINE_COLORS[2])]:
    tau = 300 * toy_band(width)            # optical depth of the band
    axes[0].plot(lam, tau, color=c, label=label)
    axes[1].plot(lam, np.exp(-tau), color=c, label=label)
for ax in axes:
    ax.axvspan(1.60, 1.75, color="0.85", zorder=0)
axes[0].set_yscale("log"); axes[0].set_ylim(1e-2, 1e3)
axes[0].set_ylabel("absorption strength\n(optical depth τ)"); axes[0].legend(loc="upper left")
axes[1].set_ylabel("fraction of light\nthat gets through"); axes[1].set_xlabel("Wavelength [µm]")
axes[1].text(1.40, 0.06, "core:\nalmost no light\ngets through", ha="center", va="bottom", fontsize=9)
axes[1].annotate("wings: weaker absorption\nspreading out to the sides", (1.52, 0.6), (1.77, 0.3),
                 fontsize=9, arrowprops=dict(arrowstyle="->"))
axes[0].text(1.675, 100, "a neighbouring\nwindow", ha="center", va="center", fontsize=9, color="0.3")
axes[0].set_title("TOY: one absorption band, its core and wings", fontsize=10);

The core is so strong that it lets almost nothing through at either pressure. The wings are what change: at high
pressure they reach further and take out part of the light in the neighbouring window. Notebook 3 shows that this
is why the height of each window peak depends on the bands next to it.

### Contribution functions
The τ = 2/3 level is a shorthand. The **contribution function** shows *how much* light at each
wavelength comes from each layer (dark = most). The cyan line is the τ = 2/3 level.

In [ ]:
fig, ax = bd.plot_contribution_function(std)

### Check: does the flux really tell you the temperature at τ = 2/3?
Look up the temperature at each wavelength's τ = 2/3 level and compare it with the toy spectrum's
brightness temperature (notebook 1).

In [ ]:
t_at_photosphere = np.interp(np.log(std.photosphere_bar), np.log(std.pressure_bar), std.temperature_k)
fig, ax = plt.subplots()
ax.plot(std.wave_um, std.brightness_temp_k, label="brightness temperature of the toy spectrum")
ax.plot(std.wave_um, t_at_photosphere, "--", label="temperature at the τ = 2/3 level")
ax.invert_yaxis()     # flipped, as in notebook 1: down = hotter = deeper
ax.set_xlabel("Wavelength [µm]"); ax.set_ylabel("temperature [K]\n(flipped: down = hotter = deeper)"); ax.legend()
ax.set_title("TOY: brightness temperature vs temperature at τ = 2/3", fontsize=10);

Over most of the spectrum the two curves track each other closely: **brightness temperature ≈ temperature of
the layer you see.** That is the bridge between notebook 1 and the rest of these notebooks.

Where they disagree — the short-wavelength windows, below about 2 µm — the spectrum is *hotter* than the τ = 2/3
layer. Notebook 1 explains why: on the steep side of the Planck curve, slightly deeper (hotter) layers are so much
brighter that they contribute more than their share, so the light comes from a bit below τ = 2/3. You can see this
in the contribution function above: in those windows the dark band sits below the cyan line. "Flux tells you the
temperature at τ = 2/3" is a rule of thumb, not an exact law.

## 3. Thought experiment: an isothermal atmosphere

**Predict before running:** the "isothermal" toy has the *same absorbers* as the standard one, but
every layer has the same temperature (1000 K). What will its spectrum look like? Will the bands show?

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for t in (std, iso):
    axes[0].plot(t.wave_um, t.flux, label=t.name)
    axes[1].plot(t.wave_um, t.brightness_temp_k, label=t.name)
axes[0].plot(std.wave_um, bd.blackbody_surface_flux(std.wave_um, 1000), "k:", label="1000 K blackbody")
axes[0].set_yscale("log"); axes[0].set_ylabel(r"$F_\lambda$ (toy)"); axes[0].legend()
axes[1].invert_yaxis()
axes[1].set_ylabel("brightness temperature [K]\n(flipped: down = hotter)"); axes[1].set_xlabel("Wavelength [µm]")
axes[0].set_title("TOY: standard vs isothermal", fontsize=10);

No features at all — the isothermal toy is exactly a 1000 K blackbody, even though its absorbers are
just as strong.

**Why?** Follow the light at a band wavelength as it climbs out. Light from a deep layer is partly absorbed by the
layer above it. But a layer that absorbs at some wavelength also **emits** at that wavelength, and how much it emits
depends only on its temperature (the Planck curve, notebook 1). If that layer has the *same* temperature as the one
below, it emits exactly as much light as it absorbed: the light the absorber took out is replaced, one for one.
Repeat that layer by layer and nothing is lost, however strong the absorber.

Now let the temperature fall with height, as in the standard toy. Each layer is cooler than the one below, so it
emits *less* than it absorbs. In a band we see a high, cool layer, so less light comes out than in a window, where we
see a deep, hot layer — a trough. The bigger the temperature difference between those two layers, the deeper the band.

So absorption only makes a **band** when the layer you see in the band has a *different temperature* from the layer
you see in the window. **Spectral features need a temperature gradient.**

## 4. Thought experiment: a thermal inversion

**Predict:** in the "inversion" toy, the temperature *rises* again high in the atmosphere (above ~30 mbar).
What happens to the strongest bands, whose τ = 2/3 level sits in that warm upper layer?

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for t in (std, inv):
    axes[0].plot(t.wave_um, t.flux, label=t.name)
    axes[1].plot(t.wave_um, t.brightness_temp_k, label=t.name)
axes[0].set_yscale("log"); axes[0].set_ylabel(r"$F_\lambda$ (toy)"); axes[0].legend()
axes[1].invert_yaxis()
axes[1].set_ylabel("brightness temperature [K]\n(flipped: down = hotter)"); axes[1].set_xlabel("Wavelength [µm]")
axes[0].set_title("TOY: standard vs inversion", fontsize=10);

The strongest bands flip from **absorption** (dips) to **emission** (bumps), because the layer we see
in them is now *hotter* than the layer we see in the windows. Weaker bands, which form deeper (below the
inversion), stay in absorption.

**A real example.** Rustamkulov et al. (2026) report a weak **3.3 µm CH<sub>4</sub> emission feature** in their
T7 spectrum, possibly from a thermal inversion. The next cell shows it once the T7 reference is installed.

In [ ]:
t7 = bd.optional(bd.load_reference, "T7")
if t7 is not None:
    bd.plot_spectrum(t7.between(2.8, 4.2), log=True, title="T7 reference near 3.3 um (look for a small bump)")

## 5. Adding absorber: deeper, then saturated

> 🧸 **TOY MODEL.** The atmosphere in this notebook is a deliberately simple toy. Its bands sit at roughly the right wavelengths so the plots look familiar, but their shapes and strengths are made up. Use it for *ideas*, never for numbers about real objects.

Below, the toy's CH<sub>4</sub> bands are multiplied by factors from 0 (none) to 1000.
**Predict:** if you multiply the CH<sub>4</sub> absorption by 10, does the 3.3 µm band get 10 times deeper?

In [ ]:
sw = bd.load_toy("ch4_sweep")
fig, ax = plt.subplots()
colors = plt.cm.viridis(np.linspace(0, 0.9, len(sw.multipliers)))
for m, f, c in zip(sw.multipliers, sw.spherex_flux, colors):
    ax.plot(sw.spherex_wave_um, f, "o-", ms=2, color=c, label=f"CH4 x {m:g}")
ax.set_xlim(2.6, 4.4); ax.set_yscale("log"); ax.legend(ncol=2, fontsize=8)
ax.set_xlabel("Wavelength [µm]"); ax.set_ylabel(r"$F_\lambda$ (toy, SPHEREx-like channels)")
ax.set_title("TOY: CH4 multiplied by each factor", fontsize=10);

In [ ]:
i33 = np.argmin(abs(sw.wave_um - 3.32))
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(sw.multipliers[1:], sw.brightness_temp_k[1:, i33], "o-")
ax.axhline(sw.brightness_temp_k[0, i33], color="0.5", ls=":", label="no CH4")
ax.invert_yaxis()     # down = hotter = deeper, so the curve rises as we see higher, cooler layers
ax.set_xlabel("CH4 multiplier (log scale)"); ax.set_ylabel("brightness temperature at 3.32 µm [K]\n(flipped: down = hotter)")
ax.legend(); ax.set_title("TOY: band depth vs amount of absorber", fontsize=10);

Two lessons, both true of real atmospheres too:

1. **Roughly logarithmic growth.** Each *factor* of 10 more absorber raises the τ = 2/3 level by about
   the same step in log P, so the band deepens by similar steps per factor of 10 — not in proportion.
2. **Saturation.** Once the τ = 2/3 level reaches the cool, nearly isothermal upper atmosphere, adding
   more absorber barely changes anything: the band is **saturated**. Strong real bands — CH<sub>4</sub> at 3.3 µm in
   cold dwarfs, for example — barely respond to changes in absorber amount.

That is why in these notebooks we say a band shows **more absorption**, not "more abundance": a deeper band
can come from more molecules, a different temperature profile, or both, and a saturated band can hide a
big change in the amount of a molecule.

### Exercise 2.1
In the contribution-function plot, find a wavelength whose light comes from the deepest layers. What
kind of region of the spectrum is it — a band or a window?

<details><summary><b>Answer</b> (try first, then click)</summary>

The deepest light comes out in the windows between bands (e.g. near 1.1, 1.27, 1.6, 2.1 and 4.0 µm in
the toy), where there is the least absorption. In real brown dwarfs, these are the Y, J, H, K and 4 µm peaks.

</details>

### Exercise 2.2
In the isothermal toy, the light at 3.3 µm still comes from much higher up than at 4.0 µm. So why is
there no band?

<details><summary><b>Answer</b> (try first, then click)</summary>

Because every layer has the same temperature, so B<sub>λ</sub>(T) is the same wherever the light comes
from. A band only appears when the layer you see in the band has a different temperature from the one you
see in the window.

</details>

### Exercise 2.3
From the band-depth plot, roughly how much does T<sub>b</sub> at 3.32 µm drop going from ×0.01 to ×0.1?
From ×10 to ×100?

<details><summary><b>Answer</b> (try first, then click)</summary>

Tens of kelvin for the first step and almost nothing for the second (read the exact numbers off your
plot). The band is saturated at large multipliers.

</details>

## Summary

- There is no surface: we see light from the layer where τ ≈ 2/3, and that depth depends on wavelength.
- The flux at a wavelength tells you the approximate temperature at that wavelength's τ = 2/3 level.
- τ depends on wavelength because each molecule absorbs in its own bands. A band has a strong core and weaker wings; pressure broadening spreads the wings further.
- No temperature gradient → no features: each layer re-emits what it absorbs. A thermal inversion turns strong bands into emission.
- Band depth grows roughly with the log of the absorber amount, then saturates.
- Field guide: [Glossary](https://github.com/zshakespear/bd-spectra-onramp/blob/main/field_guide/glossary.md)

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. What does τ = 2/3 mean physically?
<details><summary><b>Answer</b> (try first, then click)</summary>

The depth from which a typical escaping photon comes; deeper light is mostly absorbed on the way out.

</details>

2. Why does an isothermal atmosphere show no absorption bands?
<details><summary><b>Answer</b> (try first, then click)</summary>

Every layer emits exactly as much light as it absorbs (all at the same B<sub>λ</sub>(T)), so it doesn't matter which layer you see.

</details>

3. What turns an absorption band into an emission feature?
<details><summary><b>Answer</b> (try first, then click)</summary>

A thermal inversion: the layer seen in the band is hotter than the layer seen in the windows.

</details>

4. Why might a band barely change when a molecule's amount changes by 10×?
<details><summary><b>Answer</b> (try first, then click)</summary>

It is saturated.

</details>

**Next:** [Notebook 3 · The band atlas](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/03_band_atlas.ipynb)